##### Jan 12 2025

### Research Notes

The load seems to be slightly higher on average, even though the peaks are pretty close. This may be from giving such high weight to the top 5% of load hours while training the load model. We feel this is an acceptable tradeoff. The reliability of the power system is governed by extreme weather events that either drastically increase demand for power or severely reduce the available supply from generators. Therefore, we felt it important to give more weight in training to high demand days, as these are most impactful on power system reliability. However, for other tasks such as estimating the long-run average cost to operate the power system, this approach is not good. Our average cost is 153 million compared with 118 million per year. Still, I feel that this load model is acceptable, especially considering that we *trained the model on different load data* than what is in the GMLC system. Therefore it is entirely possible that we would differ by a few hundred megawatts in various quantiles. I am happy to proceed with this data set at this point!

### Next Steps...

I have the reliability simulation in place from last year (hell yeah!). All I must do now is start the machine learning and reliability testing.

The reliability testing is the easiest thing to do. I have already done my validation / comparison against the 2006 load - everything checks out. I am happy to trust the wind and solar data I have from NREL NSRDB / WTK-LED. First - I take a moment to appreciate that I've achieved a major milestone! I have finished making a 20 year-long high quality power systems data set! And a working reliability model to go with it!

Next steps are:

* Adjust system to have LOLE of ~1 unserved event/year.
* Set up method to adjust / save system configurations.
* Simulate power system for 20 years; save results (cost, LMP data, unserved energy)
* Set up learning environment: split data into train/validate/test data sets (see below)
* Input to neural network: weather data at 72 buses + 2 lag/lead terms (-2 hours, +2 hours)
* Output from neural network: load, wind, solar at each bus
* Download additional ERA5 data inputs?

#### Learning Environment - Training/Validation/Testing

Now we must decide how to do training/validation/testing. We have 18-20 years of data. We shall reserve 30% (6-8 years) of the data for testing, and use 70% (12-14 years) for training/validation.
With the 12-14 years of training/validation data, we shall do cross-validation with 2 year increments (6-7 folds). Therefore, we train each model with 10 years of ERA5 data and power system data.

### Experiment Outline

Here is an interesting experiment idea! Test the model's prediction abilities for different reliability levels of a power system, by *adding perfect capacity to it!* This is the perfect place to start introducing ideas like perfect capacity! Bit by bit in each project/paper, so it all builds off each other and you start to create a new area to explore.

It is also interesting to consider the performance under different types of power systems - e.g. solar + storage + LDES, coal and gas heavy, etc.

For this project, we need like 8 different power systems to test out. E.g., we test 5 different levels of perfect capacity for the base power system (adding/subtracting flat load, essentially), making the system more/less reliable. Heavy solar + storage + LDES system. (Recall: we removed hydro even though it is weather dependent. We also do not consider thermal generator outages, as they are not weather-dependent and would simply add noise to our results. However, for completeness we should model thermal outages in at least one run). Gas + nuclear heavy system, with some renewables + storage.

We also need to compare, e.g., different data sets, different learning approaches. An unavoidable question is: how does ERA5 itself compare with NREL data? I should take a cursory look at this. Not saying which is better/worse, but how are they different? These factors themselves may influence some of our results -- indeed, we hope to learn a mapping from one data set to the other, so understanding their differences is key. But, a detailed study on the implications for power systems analysis in the Southwest is far beyond the scope of this paper (and maybe can be the scope of future work). In particular, we do not have to address here whether ERA5 data is itself of sufficient quality for detailed power systems analysis compared with NREL data. Here, we are interested in learning to map weather data to load and generation data using machine learning, which is indeed common in industry (see E3's RECAP model). But, **we must be able to say how much our model is actually improving the accuracy of ERA5! That is the problem we are trying to solve. Therefore, we must have an ERA5 baseline to compare with...**

At the end of the day, all that matters for our work is to compare strong baselines to our model. So, not only do we want to do regular supervised ML with a deep neural network (which is a strong baseline), but perhaps also try out a few time series models from `darts`. All of these methods, however, do not use the structure of the power system optimization problem to enhance the model.

##### Idea:

We can also try a simplified objective: penalizing net load prediction error instead of load, wind, solar prediction error separately. This would just be a linear transformation in the final layer of the network. Literally multiplying $(1, -1, -1, \cdots, -1)$ with $(l, \mathbf{u})$ at each bus.

Furthermore, we can just try weighting samples during training with their LMP - i.e., we simulate the power system with the training data (targets), then weight samples with their LMP during training. This *still* isn't the same thing as our method, though perhaps is similar in spirit.